# Análise do benchmark: Hyperledger Fabric em Raspberry Pi (telessaúde emergencial)

Notebook para o Google Colab. Lê os resultados gerados por `caliper-workspace/experiment/run-experiment.sh` e produz
figuras prontas para o TCC/artigo (PNG 300 dpi e PDF vetorial).

**Como usar**
1. No Mac, compacte a pasta do experimento: `cd caliper-workspace/results && zip -r initial-wifi.zip initial-wifi`
2. No Colab: *Ambiente de execução → Executar tudo*. A segunda célula pede o `.zip` (ou use o Drive).
3. As figuras ficam em `figs/` e a última célula baixa tudo em um `.zip`.

Idioma dos gráficos: mude `LANG` na primeira célula (`'pt'` ou `'en'`).

In [ ]:
import glob, io, json, math, os, re, zipfile
import numpy as np
import pandas as pd
import matplotlib as mpl
import matplotlib.pyplot as plt
from matplotlib.ticker import FuncFormatter, NullLocator

# ----------------------------------------------------------------------------- configuration
LANG = 'pt'                 # 'pt' (Portuguese) or 'en' (English)
OUT_DIR = 'figs'
BATCH_TIMEOUT_MS = 2000     # orderer BatchTimeout of the channel (configtx.yaml)
os.makedirs(OUT_DIR, exist_ok=True)

OPS = ['insert', 'update', 'read', 'history', 'mixed']

TXT = {
    'pt': {
        'offered': 'Taxa oferecida (TPS)', 'throughput': 'Vazão medida (TPS)', 'latency': 'Latência (ms)',
        'ideal': 'Ideal (vazão = oferecida)', 'saturated_pt': 'Passo saturado', 'saturated_zone': 'saturação',
        'batch_timeout': 'BatchTimeout do orderer (2 s)', 'fail_pct': 'Falhas (%)', 'cpu': 'CPU média (%)',
        'temp': 'Temperatura máxima (°C)', 'sd_write': 'Escrita no cartão SD (kB/s)', 'time_s': 'Tempo desde o início do passo (s)',
        'ecdf_y': 'Fração das transações com latência ≤ x', 'throttle': 'redução de frequência (~80 °C)',
        'sustainable': 'Sustentável (TPS)', 'first_sat': 'Primeira saturação (TPS)', 'p95_at': 'p95 no sustentável (ms)',
        'bottleneck': 'Gargalo (CPU no passo saturado)', 'operation': 'Operação', 'none': 'não atingida',
        'rtt_p95': 'RTT ICMP p95 durante o passo (ms)', 'rtt_max': 'RTT máximo', 'step_order': 'Passos em ordem cronológica',
        'op_insert': 'Criar (insert)', 'op_update': 'Atualizar (update)', 'op_read': 'Ler (read)',
        'op_history': 'Histórico (auditoria)', 'op_mixed': 'Misto 60/20/10/10',
        't1_title': 'Vazão medida × taxa oferecida',
        't1_sub': 'Escritas (quentes) saturam; leituras acompanham a diagonal. Círculos vermelhos: passos saturados.',
        't2_title': 'Latência por operação em função da carga',
        't2_sub': 'Linhas: p50, p95 e p99. Faixa: IC 95% entre repetições. Região vermelha: saturação.',
        't3_title': 'Caminho de escrita (insert): latência e falhas',
        't3_sub': 'Com pouca carga a transação espera o bloco fechar (≈ BatchTimeout); com muita, forma-se fila.',
        't4_title': 'Distribuição da latência de escrita (ECDF)',
        't4_sub': 'Baixa carga: espalhada entre 0 e 2 s (espera pelo bloco). Alta carga: desloca-se para a direita.',
        't5_title': 'Formação de fila durante o passo',
        't5_sub': 'Cada ponto é uma transação. Estável: latência plana. Saturado: a latência sobe ao longo do passo.',
        't6_title': 'Uso de recursos dos nós (insert)',
        't6_sub': 'O Pi 3 (peer da emergência e gateway) é o primeiro a esgotar CPU.',
        't7_title': 'Leitura × escrita: latência mediana (p50)',
        't7_sub': 'Leituras não passam por endosso, ordenação nem commit: duas ordens de grandeza mais rápidas.',
        't8_title': 'Resumo de capacidade por operação',
        't8_sub': 'Taxa sustentável = maior passo antes da primeira saturação (mediana das repetições válidas).',
        't9_title': 'Série temporal dos nós no passo saturado',
        't9_sub': 'Faixa cinza: janela medida. A CPU do Pi 3 encosta no limite enquanto a latência cresce.',
        't10_title': 'Qualidade da rede durante os testes (Wi-Fi)',
        't10_sub': 'RTT p95 por passo. Picos indicam jitter do Wi-Fi, uma limitação do ambiente de teste.',
    },
    'en': {
        'offered': 'Offered rate (TPS)', 'throughput': 'Measured throughput (TPS)', 'latency': 'Latency (ms)',
        'ideal': 'Ideal (throughput = offered)', 'saturated_pt': 'Saturated step', 'saturated_zone': 'saturation',
        'batch_timeout': 'Orderer BatchTimeout (2 s)', 'fail_pct': 'Failures (%)', 'cpu': 'Mean CPU (%)',
        'temp': 'Peak temperature (°C)', 'sd_write': 'SD card writes (kB/s)', 'time_s': 'Time since the step started (s)',
        'ecdf_y': 'Fraction of transactions with latency ≤ x', 'throttle': 'frequency throttling (~80 °C)',
        'sustainable': 'Sustainable (TPS)', 'first_sat': 'First saturation (TPS)', 'p95_at': 'p95 at sustainable (ms)',
        'bottleneck': 'Bottleneck (CPU at saturated step)', 'operation': 'Operation', 'none': 'not reached',
        'rtt_p95': 'ICMP RTT p95 during the step (ms)', 'rtt_max': 'Max RTT', 'step_order': 'Steps in chronological order',
        'op_insert': 'Create (insert)', 'op_update': 'Update', 'op_read': 'Read',
        'op_history': 'History (audit)', 'op_mixed': 'Mixed 60/20/10/10',
        't1_title': 'Measured throughput vs offered rate',
        't1_sub': 'Writes (warm colors) saturate; reads follow the diagonal. Red circles: saturated steps.',
        't2_title': 'Latency per operation as a function of load',
        't2_sub': 'Lines: p50, p95 and p99. Band: 95% CI across repetitions. Red region: saturation.',
        't3_title': 'Write path (insert): latency and failures',
        't3_sub': 'At low load a transaction waits for the block to be cut (~BatchTimeout); at high load a queue forms.',
        't4_title': 'Write latency distribution (ECDF)',
        't4_sub': 'Low load: spread between 0 and 2 s (waiting for the block). High load: shifts to the right.',
        't5_title': 'Queue build-up during the step',
        't5_sub': 'Each dot is a transaction. Stable: flat latency. Saturated: latency climbs during the step.',
        't6_title': 'Node resource usage (insert)',
        't6_sub': 'The Pi 3 (emergency peer and gateway) is the first to run out of CPU.',
        't7_title': 'Read vs write: median latency (p50)',
        't7_sub': 'Reads skip endorsement, ordering and commit: two orders of magnitude faster.',
        't8_title': 'Capacity summary per operation',
        't8_sub': 'Sustainable rate = highest step before the first saturation (valid repetitions).',
        't9_title': 'Node time series during the saturated step',
        't9_sub': 'Gray band: measured window. The Pi 3 CPU hits the ceiling while latency grows.',
        't10_title': 'Network quality during the tests (Wi-Fi)',
        't10_sub': 'RTT p95 per step. Spikes show Wi-Fi jitter, a limitation of the test environment.',
    },
}
def L(key):
    return TXT[LANG][key]
def OPL(op):
    return TXT[LANG]['op_' + op]

# Okabe-Ito palette (color-blind safe). Writes in warm colors, reads in cool colors.
OP_COLOR = {'insert': '#D55E00', 'update': '#E69F00', 'read': '#0072B2', 'history': '#56B4E9', 'mixed': '#009E73'}
NODE_COLOR = {'pi3': '#CC79A7', 'pi5': '#0072B2', 'mac': '#7F7F7F'}
NODE_LABEL = {'pi3': 'Pi 3 B (peer emergency)', 'pi5': 'Pi 5 (peer hospital + orderer1)', 'mac': 'Mac (peer audit + 2 orderers + Caliper)'}
RED = '#B00020'

mpl.rcParams.update({
    'figure.dpi': 110, 'savefig.dpi': 300, 'savefig.bbox': 'tight',
    'font.size': 10, 'axes.titlesize': 11.5, 'axes.titleweight': 'bold', 'axes.labelsize': 10,
    'axes.spines.top': False, 'axes.spines.right': False,
    'axes.grid': True, 'grid.color': '#E4E4E4', 'grid.linewidth': 0.7, 'axes.axisbelow': True,
    'legend.frameon': False, 'legend.fontsize': 9, 'lines.linewidth': 2, 'lines.markersize': 6,
    'xtick.color': '#333333', 'ytick.color': '#333333', 'axes.edgecolor': '#555555',
})

def save(fig, name):
    fig.savefig(f'{OUT_DIR}/{name}.png')
    fig.savefig(f'{OUT_DIR}/{name}.pdf')
    plt.show()

def titles(fig, key, y=1.02):
    fig.suptitle(L(key + '_title'), x=0.01, y=y, ha='left', fontsize=13, fontweight='bold')
    fig.text(0.01, y - 0.045, L(key + '_sub'), ha='left', va='top', fontsize=9.5, color='#555555')

## 1. Carregar os dados

Envie o `.zip` da pasta do experimento (ex.: `initial-wifi.zip`). Fora do Colab, defina `RESULTS_DIR`.

In [ ]:
RESULTS_DIR = None   # outside Colab: set to the folder that contains rep1/, rep2/, steps.csv ...

def find_root(base):
    # the folder that has rep<N> sub-folders
    for dirpath, dirnames, _ in os.walk(base):
        if any(re.fullmatch(r'rep\d+', d) for d in dirnames):
            return dirpath
    raise FileNotFoundError('no rep<N> folders found: upload the zip of results/<experiment>')

if RESULTS_DIR is None:
    from google.colab import files
    uploaded = files.upload()
    for name, data in uploaded.items():
        if name.lower().endswith('.zip'):
            with zipfile.ZipFile(io.BytesIO(data)) as z:
                z.extractall('/content/results')
    RESULTS_DIR = find_root('/content/results')
else:
    RESULTS_DIR = find_root(RESULTS_DIR)
print('results folder:', RESULTS_DIR)

In [ ]:
def load_steps(root):
    csv_path = os.path.join(root, 'steps.csv')
    if os.path.exists(csv_path):
        df = pd.read_csv(csv_path)
    else:   # fall back to the per-step JSON written by the driver
        rows = [json.load(open(p)) for p in glob.glob(os.path.join(root, 'rep*', '*', 'step.json'))]
        df = pd.DataFrame(rows)
    for col in ('valid', 'saturated', 'severe'):
        if col in df:
            df[col] = df[col].astype(str).str.lower().eq('true')
    df['tps'] = pd.to_numeric(df['tps'])
    return df

steps = load_steps(RESULTS_DIR)
ops_present = [o for o in OPS if o in set(steps['op'])]
print(f'{len(steps)} steps, {steps["valid"].sum()} valid, repetitions: {sorted(steps["rep"].unique())}')
print('excluded:', steps.loc[~steps['valid'], ['rep', 'op', 'tps']].values.tolist())

T95 = {1: 12.706, 2: 4.303, 3: 3.182, 4: 2.776, 5: 2.571, 6: 2.447, 7: 2.365, 8: 2.306, 9: 2.262, 10: 2.228}
def ci95(values):
    v = np.asarray([x for x in values if x == x], dtype=float)
    if len(v) < 2:
        return np.nan
    return T95.get(len(v) - 1, 1.96) * v.std(ddof=1) / math.sqrt(len(v))

METRICS = ['throughput', 'send_tps', 'lat_p50', 'lat_p95', 'lat_p99', 'fail_pct', 'pi3_cpu_mean', 'pi5_cpu_mean',
           'mac_cpu_mean', 'pi3_temp_max', 'pi5_temp_max', 'pi3_sd_write_kbs', 'pi5_sd_write_kbs',
           'rtt_pi3_p95_ms', 'rtt_pi5_p95_ms']

def summarize(valid):
    rows = []
    for (op, tps), g in valid.groupby(['op', 'tps']):
        r = {'op': op, 'tps': tps, 'n': len(g), 'saturated': bool(g['saturated'].mean() >= 0.5)}
        for m in METRICS:
            if m in g:
                r[m] = g[m].mean()
                r[m + '_ci'] = ci95(g[m].dropna().values)
        rows.append(r)
    return pd.DataFrame(rows).sort_values(['op', 'tps']).reset_index(drop=True)

S = summarize(steps[steps['valid']])
S

In [ ]:
# helpers shared by the figures
def step_dir(root, row):
    tps = int(row['tps']) if float(row['tps']).is_integer() else row['tps']
    return os.path.join(root, row['rep'], f"{row['op']}-{tps}")

def pick_step(op, tps):
    g = steps[(steps['op'] == op) & (steps['tps'] == tps) & steps['valid']].sort_values('rep')
    return None if g.empty else g.iloc[0]

def load_tx(row):
    # per-transaction latencies of the measured round of one step
    d = step_dir(RESULTS_DIR, row)
    meta = json.load(open(os.path.join(d, 'meta.json')))
    measured = [r['index'] for r in meta['rounds'] if not r['warmup']][0]
    parts = [pd.read_csv(p) for p in glob.glob(os.path.join(d, 'latencies-w*.csv'))]
    tx = pd.concat(parts, ignore_index=True)
    tx = tx[tx['round'] == measured].sort_values('start_ms')
    tx['t_s'] = (tx['start_ms'] - tx['start_ms'].min()) / 1000.0
    return tx

def load_samples(row, node):
    p = os.path.join(step_dir(RESULTS_DIR, row), f'samples-{node}.csv')
    df = pd.read_csv(p)
    df = df[pd.to_numeric(df['ts_ms'], errors='coerce').notna()].copy()
    df['ts_ms'] = df['ts_ms'].astype('int64')
    return df

def ladder_axis(ax, values):
    ax.set_xscale('log')
    ax.set_xticks(sorted(values))
    ax.xaxis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
    ax.xaxis.set_minor_locator(NullLocator())
    ax.set_xlim(min(values) * 0.8, max(values) * 1.25)

def shade_saturation(ax, g):
    sat = g[g['saturated']]
    if sat.empty:
        return
    first = sat['tps'].min()
    before = g[g['tps'] < first]['tps']
    start = math.sqrt(before.max() * first) if len(before) else first / 1.2
    ax.axvspan(start, ax.get_xlim()[1], color=RED, alpha=0.07, lw=0, zorder=0)

def capacity_table(summary):
    rows = []
    for op in [o for o in OPS if o in set(summary['op'])]:
        g = summary[summary['op'] == op].sort_values('tps')
        sat = g[g['saturated']]
        first_sat = sat['tps'].min() if len(sat) else np.nan
        ok = g[~g['saturated']] if math.isnan(first_sat) else g[g['tps'] < first_sat]
        sustainable = ok['tps'].max() if len(ok) else np.nan
        ref = g[g['tps'] == first_sat].iloc[0] if len(sat) else g.iloc[-1]
        cpu = {'Pi 3': ref.get('pi3_cpu_mean', np.nan), 'Pi 5': ref.get('pi5_cpu_mean', np.nan), 'Mac': ref.get('mac_cpu_mean', np.nan)}
        node = max(cpu, key=lambda k: -1 if cpu[k] != cpu[k] else cpu[k])
        p95 = g[g['tps'] == sustainable]['lat_p95'].iloc[0] if not math.isnan(sustainable) else np.nan
        rows.append({'op': op, 'sustainable': sustainable, 'first_sat': first_sat, 'p95': p95,
                     'node': node, 'cpu': cpu[node]})
    return pd.DataFrame(rows)

CAP = capacity_table(S)
CAP

## 2. Vazão e latência

In [ ]:
# Figure 1: throughput vs offered rate
fig, ax = plt.subplots(figsize=(6.6, 5.4))
lim = [S['tps'].min() * 0.8, S['tps'].max() * 1.25]
ax.plot(lim, lim, ls='--', color='#8A8A8A', lw=1.2, label=L('ideal'), zorder=1)
for op in ops_present:
    g = S[S['op'] == op].sort_values('tps')
    ax.errorbar(g['tps'], g['throughput'], yerr=g['throughput_ci'].fillna(0), color=OP_COLOR[op], marker='o',
                capsize=3, label=OPL(op), zorder=3)
    sat = g[g['saturated']]
    ax.scatter(sat['tps'], sat['throughput'], s=150, facecolors='none', edgecolors=RED, linewidths=1.8, zorder=4)
ax.scatter([], [], s=150, facecolors='none', edgecolors=RED, linewidths=1.8, label=L('saturated_pt'))
ax.set_xscale('log'); ax.set_yscale('log')
ticks = sorted(S['tps'].unique())
for axis in (ax.xaxis, ax.yaxis):
    axis.set_major_formatter(FuncFormatter(lambda v, _: f'{v:g}'))
    axis.set_minor_locator(NullLocator())
ax.set_xticks(ticks); ax.set_yticks(ticks)
ax.set_xlim(lim); ax.set_ylim(lim)
ax.set_xlabel(L('offered')); ax.set_ylabel(L('throughput'))
ax.legend(loc='upper left')
titles(fig, 't1', y=1.10)
save(fig, 'fig1_throughput')

In [ ]:
# Figure 2: latency percentiles per operation (small multiples)
n = len(ops_present); cols = 3; rows = math.ceil(n / cols)
fig, axes = plt.subplots(rows, cols, figsize=(12.8, 3.9 * rows), squeeze=False)
for ax, op in zip(axes.ravel(), ops_present):
    g = S[S['op'] == op].sort_values('tps')
    ladder_axis(ax, list(g['tps']))
    for col, ls, name in (('lat_p50', '-', 'p50'), ('lat_p95', '--', 'p95'), ('lat_p99', ':', 'p99')):
        ax.plot(g['tps'], g[col], ls=ls, marker='o', ms=4.5, color=OP_COLOR[op], label=name)
        if col != 'lat_p99':
            ci = g[col + '_ci'].fillna(0)
            ax.fill_between(g['tps'], np.maximum(g[col] - ci, 1), g[col] + ci, color=OP_COLOR[op], alpha=0.15, lw=0)
    ax.set_yscale('log')
    shade_saturation(ax, g)
    ax.set_title(OPL(op), loc='left', fontsize=10.5)
    ax.set_xlabel(L('offered')); ax.set_ylabel(L('latency'))
    ax.legend(loc='upper left', ncol=3, fontsize=8)
for ax in axes.ravel()[n:]:
    ax.axis('off')
titles(fig, 't2', y=1.03)
fig.tight_layout()
save(fig, 'fig2_latency_per_operation')

In [ ]:
# Figure 3: write path deep-dive (latency and failures)
op = 'insert' if 'insert' in ops_present else ops_present[0]
g = S[S['op'] == op].sort_values('tps')
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4.6), gridspec_kw={'width_ratios': [1.7, 1]})
ladder_axis(ax1, list(g['tps']))
for col, ls, name in (('lat_p50', '-', 'p50'), ('lat_p95', '--', 'p95'), ('lat_p99', ':', 'p99')):
    ax1.plot(g['tps'], g[col], ls=ls, marker='o', color=OP_COLOR[op], label=name)
ax1.axhline(BATCH_TIMEOUT_MS, color='#777777', ls='-.', lw=1.2)
ax1.text(g['tps'].max() * 1.2, BATCH_TIMEOUT_MS * 1.06, L('batch_timeout'), ha='right', va='bottom', fontsize=8.5, color='#555555')
ax1.set_yscale('log')
shade_saturation(ax1, g)
ax1.set_xlabel(L('offered')); ax1.set_ylabel(L('latency')); ax1.legend(loc='lower left', ncol=3)

pos = np.arange(len(g))
colors = [RED if s else OP_COLOR[op] for s in g['saturated']]
ax2.bar(pos, g['fail_pct'], color=colors, width=0.62)
ax2.set_xticks(pos); ax2.set_xticklabels([f'{t:g}' for t in g['tps']])
ax2.set_xlabel(L('offered')); ax2.set_ylabel(L('fail_pct')); ax2.grid(axis='x', visible=False)
for x, v in zip(pos, g['fail_pct']):
    ax2.text(x, v + max(g['fail_pct'].max() * 0.02, 0.2), f'{v:.1f}', ha='center', va='bottom', fontsize=8)
titles(fig, 't3', y=1.09)
fig.tight_layout()
save(fig, 'fig3_write_path')

In [ ]:
# Figure 4: latency ECDF at increasing load (insert)
op = 'insert' if 'insert' in ops_present else ops_present[0]
available = sorted(S[S['op'] == op]['tps'].unique())
targets = [t for t in (2, 10, 20, 40) if t in available] or available[:4]
colors = plt.cm.viridis(np.linspace(0.1, 0.85, len(targets)))
fig, ax = plt.subplots(figsize=(7.2, 4.8))
for color, t in zip(colors, targets):
    row = pick_step(op, t)
    if row is None:
        continue
    tx = load_tx(row)
    x = np.sort(tx.loc[tx['ok'] == 1, 'latency_ms'].values)
    y = np.arange(1, len(x) + 1) / len(x)
    ax.step(x, y, where='post', color=color, lw=2, label=f'{t:g} TPS (n={len(x)})')
ax.axvline(BATCH_TIMEOUT_MS, color='#777777', ls='-.', lw=1.2)
ax.text(BATCH_TIMEOUT_MS * 1.02, 0.04, L('batch_timeout'), fontsize=8.5, color='#555555')
ax.set_xlim(0, 6000); ax.set_ylim(0, 1.01)
ax.set_xlabel(L('latency')); ax.set_ylabel(L('ecdf_y'))
ax.legend(loc='lower right')
titles(fig, 't4', y=1.10)
save(fig, 'fig4_latency_ecdf')

In [ ]:
# Figure 5: queue build-up inside a step (last stable, first saturated, next)
op = 'insert' if 'insert' in ops_present else ops_present[0]
cap = CAP[CAP['op'] == op].iloc[0]
available = sorted(S[S['op'] == op]['tps'].unique())
chosen = []
if not math.isnan(cap['sustainable']):
    chosen.append(cap['sustainable'])
if not math.isnan(cap['first_sat']):
    chosen.append(cap['first_sat'])
    higher = [t for t in available if t > cap['first_sat']]
    if higher:
        chosen.append(higher[0])
chosen = [t for t in chosen if pick_step(op, t) is not None]
fig, axes = plt.subplots(len(chosen), 1, figsize=(9.5, 2.9 * len(chosen)), sharex=True, sharey=True, squeeze=False)
for ax, t in zip(axes.ravel(), chosen):
    row = pick_step(op, t)
    tx = load_tx(row)
    ok = tx[tx['ok'] == 1]
    saturated = bool(row['saturated'])
    color = RED if saturated else OP_COLOR[op]
    ax.scatter(ok['t_s'], ok['latency_ms'], s=10, alpha=0.45, color=color, lw=0)
    med = ok['latency_ms'].rolling(15, center=True, min_periods=5).median()
    ax.plot(ok['t_s'], med, color='black', lw=2)
    ax.set_title(f'{t:g} TPS' + (f'  ({L("saturated_zone")})' if saturated else ''), loc='left', fontsize=10, color=color)
    ax.set_ylabel(L('latency'))
axes.ravel()[-1].set_xlabel(L('time_s'))
titles(fig, 't5', y=1.04)
fig.tight_layout()
save(fig, 'fig5_queue_buildup')

## 3. Recursos, comparação e capacidade

In [ ]:
# Figure 6: resource usage per node versus offered rate (insert)
op = 'insert' if 'insert' in ops_present else ops_present[0]
g = S[S['op'] == op].sort_values('tps')
fig, axes = plt.subplots(1, 3, figsize=(13.5, 4.3))
panels = [('cpu_mean', 'cpu', ('pi3', 'pi5', 'mac')), ('temp_max', 'temp', ('pi3', 'pi5')), ('sd_write_kbs', 'sd_write', ('pi3', 'pi5'))]
for ax, (suffix, ylabel, nodes) in zip(axes, panels):
    ladder_axis(ax, list(g['tps']))
    for node in nodes:
        col = f'{node}_{suffix}'
        if col not in g:
            continue
        ax.errorbar(g['tps'], g[col], yerr=g[col + '_ci'].fillna(0), color=NODE_COLOR[node], marker='o', capsize=3, label=NODE_LABEL[node])
    shade_saturation(ax, g)
    ax.set_xlabel(L('offered')); ax.set_ylabel(L(ylabel))
axes[0].set_ylim(0, 105)
axes[1].axhline(80, color='#777777', ls='-.', lw=1.1)
axes[1].text(g['tps'].min() * 0.85, 80.6, L('throttle'), fontsize=8, color='#555555')
axes[0].legend(loc='upper left', fontsize=8)
titles(fig, 't6', y=1.09)
fig.tight_layout()
save(fig, 'fig6_resources')

In [ ]:
# Figure 7: read vs write, median latency at the rates all operations share
common = None
for op in ops_present:
    tps_op = set(S[S['op'] == op]['tps'])
    common = tps_op if common is None else common & tps_op
common = sorted(t for t in common if t <= 20)
fig, ax = plt.subplots(figsize=(8.8, 4.8))
width = 0.8 / len(ops_present)
for i, op in enumerate(ops_present):
    g = S[(S['op'] == op) & (S['tps'].isin(common))].sort_values('tps')
    x = np.arange(len(common)) + (i - (len(ops_present) - 1) / 2) * width
    bars = ax.bar(x, g['lat_p50'], width * 0.92, color=OP_COLOR[op], label=OPL(op))
    for b, v in zip(bars, g['lat_p50']):
        ax.text(b.get_x() + b.get_width() / 2, v * 1.12, f'{v:,.0f}'.replace(',', '.') if LANG == 'pt' else f'{v:,.0f}',
                ha='center', va='bottom', fontsize=7, rotation=90)
ax.set_yscale('log'); ax.set_ylim(1, 12000)
ax.set_xticks(np.arange(len(common))); ax.set_xticklabels([f'{t:g}' for t in common])
ax.set_xlabel(L('offered')); ax.set_ylabel(L('latency') + '  (p50)')
ax.grid(axis='x', visible=False)
ax.legend(ncol=3, loc='upper center', bbox_to_anchor=(0.5, -0.17))
titles(fig, 't7', y=1.10)
save(fig, 'fig7_read_vs_write')

In [ ]:
# Figure 8: capacity summary table
def fmt(v, digits=0):
    return L('none') if v != v else f'{v:.{digits}f}'.rstrip('0').rstrip('.') if digits else f'{v:.0f}'

cell_text = []
for _, r in CAP.iterrows():
    cell_text.append([OPL(r['op']), fmt(r['sustainable']), fmt(r['first_sat']),
                      '-' if r['p95'] != r['p95'] else f"{r['p95']:.0f}", f"{r['node']} ({r['cpu']:.0f}%)"])
fig, ax = plt.subplots(figsize=(11, 0.55 * (len(cell_text) + 2)))
ax.axis('off')
tbl = ax.table(cellText=cell_text,
               colLabels=[L('operation'), L('sustainable'), L('first_sat'), L('p95_at'), L('bottleneck')],
               loc='center', cellLoc='center', colLoc='center')
tbl.auto_set_font_size(False); tbl.set_fontsize(9.5); tbl.scale(1, 1.7)
for (r, c), cell in tbl.get_celld().items():
    cell.set_edgecolor('#DDDDDD')
    if r == 0:
        cell.set_facecolor('#2B2B2B'); cell.get_text().set_color('white'); cell.get_text().set_fontweight('bold')
    elif c == 0:
        cell.get_text().set_fontweight('bold')
        cell.get_text().set_color(OP_COLOR[CAP.iloc[r - 1]['op']])
titles(fig, 't8', y=1.03)
save(fig, 'fig8_capacity_summary')

## 4. Diagnóstico do ambiente

In [ ]:
# Figure 9: node time series during the first saturated step of the write path
op = 'insert' if 'insert' in ops_present else ops_present[0]
cap = CAP[CAP['op'] == op].iloc[0]
target = cap['first_sat'] if not math.isnan(cap['first_sat']) else S[S['op'] == op]['tps'].max()
row = pick_step(op, target)
t0, t1 = float(row['t0']), float(row['t1'])
fig, axes = plt.subplots(3, 1, figsize=(10, 8.4), sharex=True)
for node in ('pi3', 'pi5', 'mac'):
    s = load_samples(row, node)
    x = (s['ts_ms'] - t0) / 1000.0
    cpu = s['cpu_pct'] if node != 'mac' else 100 - s['cpu_idle_pct']
    axes[0].plot(x, cpu, color=NODE_COLOR[node], lw=1.7, label=NODE_LABEL[node])
    if node != 'mac':
        axes[1].plot(x, s['temp_c'], color=NODE_COLOR[node], lw=1.7, label=NODE_LABEL[node])
tx = load_tx(row)
ok = tx[tx['ok'] == 1]
axes[2].scatter(ok['t_s'], ok['latency_ms'], s=9, alpha=0.5, color=RED, lw=0)
axes[2].plot(ok['t_s'], ok['latency_ms'].rolling(15, center=True, min_periods=5).median(), color='black', lw=2)
for ax in axes:
    ax.axvspan(0, (t1 - t0) / 1000.0, color='#999999', alpha=0.12, lw=0)
axes[0].set_ylabel(L('cpu')); axes[0].set_ylim(0, 105); axes[0].legend(loc='upper left', fontsize=8)
axes[1].set_ylabel(L('temp')); axes[1].axhline(80, color='#777777', ls='-.', lw=1.1)
axes[2].set_ylabel(L('latency')); axes[2].set_xlabel(L('time_s'))
axes[2].set_xlim(-45, (t1 - t0) / 1000.0 + 10)
fig.suptitle(f'{L("t9_title")}  ({OPL(op)}, {target:g} TPS)', x=0.01, y=1.0, ha='left', fontsize=13, fontweight='bold')
fig.text(0.01, 0.965, L('t9_sub'), ha='left', va='top', fontsize=9.5, color='#555555')
fig.tight_layout(rect=(0, 0, 1, 0.95))
save(fig, 'fig9_node_timeseries')

In [ ]:
# Figure 10: network quality (ICMP RTT during each step) in chronological order
order = steps.sort_values('t0').reset_index(drop=True)
fig, ax = plt.subplots(figsize=(11, 4.4))
x = np.arange(len(order))
for node in ('pi3', 'pi5'):
    p95 = order.get(f'rtt_{node}_p95_ms')
    mx = order.get(f'rtt_{node}_max_ms')
    if p95 is None:
        continue
    ax.plot(x, p95, color=NODE_COLOR[node], marker='o', lw=1.4, ms=4.5, label=f'{NODE_LABEL[node].split(" (")[0]}: p95')
    ax.scatter(x, mx, color=NODE_COLOR[node], marker='x', s=22, alpha=0.7, label=f'{NODE_LABEL[node].split(" (")[0]}: {L("rtt_max")}')
ax.set_yscale('log'); ax.set_ylim(5, max(2000, float(order[['rtt_pi3_max_ms', 'rtt_pi5_max_ms']].max().max()) * 1.3))
ax.set_ylabel(L('rtt_p95')); ax.set_xlabel(L('step_order'))
ax.set_xticks(x); ax.set_xticklabels([f"{r.op}\n{r.tps:g}" for r in order.itertuples()], fontsize=6.5)
ax.legend(ncol=2, loc='upper left')
titles(fig, 't10', y=1.10)
fig.tight_layout()
save(fig, 'fig10_network_quality')

## 5. Tabela para o texto e download

In [ ]:
# summary table ready to paste into the thesis (mean ± 95% CI over valid repetitions)
def pm(mean, ci, digits=0):
    if mean != mean:
        return '-'
    return f'{mean:.{digits}f}' if ci != ci else f'{mean:.{digits}f} ± {ci:.{digits}f}'

table = pd.DataFrame({
    'Operação' if LANG == 'pt' else 'Operation': [OPL(o) for o in S['op']],
    'TPS oferecido' if LANG == 'pt' else 'Offered TPS': S['tps'],
    'n': S['n'],
    'Vazão (TPS)' if LANG == 'pt' else 'Throughput (TPS)': [pm(m, c, 1) for m, c in zip(S['throughput'], S['throughput_ci'])],
    'p50 (ms)': [pm(m, c) for m, c in zip(S['lat_p50'], S['lat_p50_ci'])],
    'p95 (ms)': [pm(m, c) for m, c in zip(S['lat_p95'], S['lat_p95_ci'])],
    'p99 (ms)': [pm(m, c) for m, c in zip(S['lat_p99'], S['lat_p99_ci'])],
    'Falhas (%)' if LANG == 'pt' else 'Failures (%)': S['fail_pct'].round(1),
    'Saturado' if LANG == 'pt' else 'Saturated': S['saturated'],
})
table.to_csv(f'{OUT_DIR}/summary_table.csv', index=False)
table

In [ ]:
# download everything (PNG, PDF and the table)
import shutil
shutil.make_archive('figs', 'zip', OUT_DIR)
try:
    from google.colab import files
    files.download('figs.zip')
except ImportError:
    print('figures in', os.path.abspath(OUT_DIR))